# Генерация словообразовательных гнёзд через DerivBase.Ru — исправленная версия

Правки к исходному ноутбуку:
1. **Порядок ячеек**: `evaluator = Derivation()` определяется до первого использования (раньше `derive_ext` вызывался до создания объекта).
2. **Импорт**: `from src.Derivation import ...` -> `from src.derivation import ...` (регистр ломает импорт).
3. **`derive_ext`**: метод удалён из актуальной версии — добавлена совместимая функция `derive_ext`.
4. **MorphBerta** вынесена под флаг `RUN_MORPHBERTA` (нужны `simpletransformers` и каталог модели `morphberta-t`).
5. **Викисловарь** — под флагом `RUN_WIKTIONARY`, с задержкой и кэшем (тысячи запросов без пауз дают 429).
6. `pymorphy_to_derivbase[pos]` -> `.get(pos)` (не падаем на KeyError для прочих POS).
7. Исправлен путь в архиве: `derivat_families_derivbase/`.

In [ ]:
import os, sys, time
from collections import Counter
from urllib.parse import quote

SEP = chr(9)    # табуляция
NL  = chr(10)   # перевод строки

# Флаги окружения (можно переопределить перед запуском):
#   RUN_MORPHBERTA=1  MORPHBERTA_DIR=/path/to/morphberta-t
#   RUN_WIKTIONARY=1  WIKTIONARY_DELAY=0.34  LIMIT_CANDIDATES=0
RUN_MORPHBERTA   = os.environ.get('RUN_MORPHBERTA', '0') == '1'
RUN_WIKTIONARY   = os.environ.get('RUN_WIKTIONARY', '1') == '1'
WIKTIONARY_DELAY = float(os.environ.get('WIKTIONARY_DELAY', '0.34'))
LIMIT_CANDIDATES = int(os.environ.get('LIMIT_CANDIDATES', '0'))

DERIVBASE_DIR = os.environ.get('DERIVBASE_DIR', os.path.abspath('DerivBaseRu'))
if not os.path.isdir(DERIVBASE_DIR):
    DERIVBASE_DIR = os.path.abspath('.')
if DERIVBASE_DIR not in sys.path:
    sys.path.insert(0, DERIVBASE_DIR)

from src.derivation import Derivation
evaluator = Derivation()

def derive_ext(evaluator, word_b, pos_b=None, pos_a=None, **kwargs):
    res = evaluator.derive(word_b=word_b, pos_b=pos_b, pos_a=pos_a, is_extended=True, **kwargs)
    pairs = []
    for word_a, rule_ids in res.items():
        for rule_id in rule_ids:
            pairs.append((word_a, rule_id))
    return pairs

print('DerivBaseRu:', DERIVBASE_DIR)
print('пример derive_ext:', derive_ext(evaluator, word_b='любить', pos_b='verb')[:3])

## (опционально) Морфологическая разметка MorphBerta

In [ ]:
MORPHBERTA_MODEL = None
MORPHBERTA_AVAILABLE = False
if RUN_MORPHBERTA:
    try:
        from simpletransformers.ner import NERModel
        MODEL_DIR = os.environ.get('MORPHBERTA_DIR', 'morphberta-t')
        if os.path.isdir(MODEL_DIR):
            MORPHBERTA_MODEL = NERModel('roberta', MODEL_DIR, use_cuda=False)
            MORPHBERTA_AVAILABLE = True
            print('MorphBerta загружена:', MODEL_DIR)
        else:
            print('MorphBerta: каталог модели не найден:', MODEL_DIR)
    except Exception as e:
        print('MorphBerta недоступна:', type(e).__name__, e)
else:
    print('MorphBerta отключена (RUN_MORPHBERTA=0)')

In [ ]:
def convert2parsing(lemma, bmes):
    parsing = list()
    current_mtype = ''
    current_mtext = ''
    for letter, label in zip(lemma, bmes):
        pos = label[0]
        mtype = label[2:]
        if pos == 'S':
            if current_mtext:
                parsing.append({'morpheme': current_mtext, 'type': current_mtype})
            parsing.append({'morpheme': letter, 'type': mtype})
            current_mtype = ''
            current_mtext = ''
        elif pos == 'B':
            if current_mtext:
                parsing.append({'morpheme': current_mtext, 'type': current_mtype})
            current_mtext = letter
            current_mtype = mtype
        else:
            current_mtext += letter
    if current_mtext:
        parsing.append({'morpheme': current_mtext, 'type': current_mtype})
    return {lemma: parsing}

In [ ]:
def ruroberta_predict(lemma):
    import pandas as pd
    if not MORPHBERTA_AVAILABLE:
        raise RuntimeError('MorphBerta недоступна: задайте RUN_MORPHBERTA=1, MORPHBERTA_DIR и установите simpletransformers.')
    eval_data = [(0, lemma, '0')] + [(0, letter, '0') for letter in lemma]
    eval_data = pd.DataFrame(eval_data, columns=['sentence_id', 'words', 'labels'])
    result, model_outputs, preds_list = MORPHBERTA_MODEL.eval_model(eval_data)
    return convert2parsing(lemma, preds_list[0][1:])

if MORPHBERTA_AVAILABLE:
    print(ruroberta_predict('отчим'))
else:
    print('ruroberta_predict пропущен: MorphBerta не загружена')

In [ ]:
import pymorphy3
pymorph = pymorphy3.MorphAnalyzer()
print(pymorph.parse('замуж')[0])

In [ ]:
pymorphy_to_derivbase = {
    'noun': 'noun',
    'adjf': 'adj',
    'comp': 'adj',    # сравнительная степень прилагательного
    'verb': 'verb',
    'infn': 'verb',   # инфинитив (pymorphy отдаёт INFN, а не VERB)
    'advb': 'adv',
}

def get_ruroberta_morphemes(word):
    morphemes = ruroberta_predict(word)[word]
    return [(m['morpheme'], m['type']) for m in morphemes]

def get_annotation_and_derivation_type(morphemes):
    full_word = ':'.join('/'.join(morph) for morph in morphemes)
    _, types = zip(*morphemes)
    morphes_counter = Counter(types)
    if morphes_counter['ROOT'] > 1:
        annot = 'sum-suff' if morphes_counter['SUFF'] > 0 else 'sum'
    elif morphes_counter['PREF'] >= 1:
        annot = 'pref-suff' if morphes_counter['SUFF'] > 0 else 'pref'
    elif morphes_counter['SUFF'] >= 1:
        annot = 'suff'
    else:
        annot = 'undef'
    return full_word, annot

In [ ]:
try:
    from curl_cffi import requests as req
except Exception:
    import requests as req

_wd_cache = {}
def wiktionary_has(word):
    if word in _wd_cache:
        return _wd_cache[word]
    ok = False
    try:
        resp = req.get('https://ru.wiktionary.org/wiki/' + quote(word), impersonate='chrome', timeout=20)
        ok = resp.status_code == 200
    except Exception as e:
        print('wiktionary error:', word, type(e).__name__)
    _wd_cache[word] = ok
    if WIKTIONARY_DELAY:
        time.sleep(WIKTIONARY_DELAY)
    return ok

def get_real_nest(word, use_network=None):
    if use_network is None:
        use_network = RUN_WIKTIONARY
    nest = set()
    parsed = pymorph.parse(word)
    if not parsed:
        return nest
    p2d = pymorphy_to_derivbase.get(str(parsed[0].tag.POS).lower())
    if p2d is None:
        return nest
    candidates = evaluator.derive(word_b=word, pos_b=p2d)
    if LIMIT_CANDIDATES:
        candidates = dict(list(candidates.items())[:LIMIT_CANDIDATES])
    for candidate in candidates:
        if use_network and not wiktionary_has(candidate):
            continue
        if MORPHBERTA_AVAILABLE:
            morphs = get_ruroberta_morphemes(candidate)
            full_word, deriv_type = get_annotation_and_derivation_type(morphs)
        else:
            full_word, deriv_type = candidate, 'undef'
        cparsed = pymorph.parse(candidate)
        cpos = cparsed[0].tag.POS if cparsed else None
        nest.add(SEP.join([candidate, full_word, deriv_type, str(cpos)]))
    return nest

In [ ]:
words = {
    'отец': 'otets', 'мать': 'mat', 'сын': 'syn', 'дочь': 'doch',
    'день': 'den', 'ночь': 'noch', 'утро': 'utro', 'вечер': 'vecher',
    'ветер': 'veter', 'лёд': 'lyod', 'путь': 'put', 'дорога': 'doroga', 'гора': 'gora',
    'огонь': 'ogon', 'воздух': 'vozdukh', 'дождь': 'dozhd', 'мороз': 'moroz',
    'море': 'more', 'река': 'reka', 'камень': 'kamen', 'месяц': 'mesyatz',
    'добро': 'dobro', 'зло': 'zlo', 'пух': 'pukh', 'круг': 'krug', 'год': 'god',
    'муж': 'muz', 'жена': 'zhena', 'семья': 'semjya', 'трава': 'trava', 'язык': 'yazyk',
    'ребёнок': 'rebenok', 'человек': 'chelovek', 'друг': 'drug',
    'еда': 'eda', 'сердце': 'serdtse', 'глаз': 'glaz', 'колено': 'koleno', 'голова': 'golova', 'живот': 'zhivot',
    'солнце': 'solntse', 'небо': 'nebo', 'собака': 'sobaka', 'кошка': 'koshka',
    'хлеб': 'khleb', 'соль': 'sol', 'дерево': 'derevo',
    'школа': 'shkola', 'двор': 'dvor', 'образ': 'obraz', 'время': 'vremya', 'яйцо': 'yaitso', 'клетка': 'kletka', 'жизнь': 'zhizn',
    'будить': 'budit', 'шутить': 'shutit',
    'красный': 'krasnyj', 'чёрный': 'chernyj', 'белый': 'belyj', 'храбрый': 'khrabryj',
    'левый': 'levyj', 'правый': 'pravyj', 'прямой': 'pryamoj', 'кривой': 'krivoj'}

prefix = 'derivat_families_derivbase/'
os.makedirs(prefix, exist_ok=True)

for word, alias in words.items():
    path = prefix + alias + '.txt'
    if os.path.exists(path):
        print(word, 'skip')
        continue
    nest = get_real_nest(word)
    print(word, len(nest))
    with open(path, 'w', encoding='utf-8') as f:
        f.write(NL.join(sorted(nest)))

In [ ]:
! zip -r derivate_families_derivbase.zip derivat_families_derivbase/

Готово: файлы гнёзд — в `derivat_families_derivbase/`, архив — `derivate_families_derivbase.zip`.